# Akkadian-to-English Translation — Seq2Seq with Attention

**Competition:** Deep Past Initiative: Machine Translation  
**Architecture:** LSTM Seq2Seq with Bahdanau Attention, separate encoder/decoder RNNs  
**GPU Budget:** 16GB, <9 hours  

This notebook:
1. Loads and augments training data (lexicon-based paraphrasing + variations)
2. Trains a Seq2Seq model with label smoothing, gradient clipping, early stopping
3. Generates predictions via beam search
4. Outputs `submission.csv`

In [ ]:
import os
import gc
import math
import random
import logging
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

import torch
import torch.nn as nn
from sklearn.model_selection import KFold, train_test_split

# Record start time
notebook_start_time = datetime.now()
with open(str(OUTPUT_DIR / 'time.txt') if 'OUTPUT_DIR' in dir() else 'time.txt', 'w') as f:
    f.write(f'Start Time: {notebook_start_time.strftime("%Y-%m-%d %H:%M:%S")}\n')
print(f'='*60)
print(f'NOTEBOOK START TIME: {notebook_start_time.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'='*60)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Device
GPU_MEM_LIMIT_GB = 15  # Conservative limit for 16GB GPUs
if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    device = torch.device('cpu')
    print('WARNING: No GPU available, using CPU')

print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

## 1. Configuration

Architecture sized for 16GB GPU with comfortable headroom.

In [ ]:
# ── Model Architecture (sized for 16GB GPU) ──
EMBEDDING_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2
LSTM_DROPOUT = 0.3
EMBEDDING_DROPOUT = 0.1
DECODER_DROPOUT = 0.1

# ── Training ──
MAX_LEN = 180
BATCH_SIZE = 32
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
GRAD_CLIP = 1.0
NUM_EPOCHS = 50
NUM_FOLDS = 3
EARLY_STOP_PATIENCE = 8
OVERFITTING_THRESHOLD = 1.25

# ── Data Augmentation ──
AUGMENT_MULTIPLIER = 6.25

# ── Inference ──
BEAM_WIDTH = 5
BEAM_TEMPERATURE = 1.0

# ── Paths (Kaggle) ──
INPUT_DIR = Path('/kaggle/input/deep-past-initiative-machine-translation')
OUTPUT_DIR = Path('/kaggle/working')

# Fallback for local testing
if not INPUT_DIR.exists():
    WORKSPACE = Path('/home/swatson/work/MachineLearning/kaggle/DeepPastChallengeTranslateAkkadianEnglish')
    INPUT_DIR = WORKSPACE / 'data' / 'raw'
    OUTPUT_DIR = WORKSPACE / 'jupyter'

print(f'Input dir: {INPUT_DIR}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Estimated model params: ~{2 * (EMBEDDING_DIM * HIDDEN_DIM * 4 * NUM_LAYERS * 4 + HIDDEN_DIM * 2 * 18000) / 1e6:.1f}M')

## 2. Data Loading

In [ ]:
# Load training data
train_df = pd.read_csv(INPUT_DIR / 'train.csv')
test_df = pd.read_csv(INPUT_DIR / 'test.csv')

print(f'Training samples: {len(train_df)}')
print(f'Test samples: {len(test_df)}')
print(f'\nTrain columns: {list(train_df.columns)}')
print(f'Test columns: {list(test_df.columns)}')

# Preview
print(f'\n--- Train Sample ---')
print(f'Akkadian: {train_df["transliteration"].iloc[0][:100]}...')
print(f'English:  {train_df["translation"].iloc[0][:100]}...')
print(f'\n--- Test Sample ---')
print(f'Akkadian: {test_df["transliteration"].iloc[0][:100]}...')

## 3. Data Augmentation

Expand the small 1561-sample training set using paraphrasing, variations, and segment combination.

In [ ]:
class DataAugmentor:
    """Augment training data using lexicon-based paraphrasing and variations."""
    
    def __init__(self, input_dir: Path):
        self.input_dir = input_dir
        self.lexicon = self._load_lexicon()
    
    def _load_lexicon(self):
        """Load OA_Lexicon_eBL.csv and build translation mappings."""
        lexicon_path = self.input_dir / 'OA_Lexicon_eBL.csv'
        if not lexicon_path.exists():
            logger.warning(f'Lexicon not found at {lexicon_path}')
            return {}
        
        lex_df = pd.read_csv(lexicon_path)
        mappings = defaultdict(set)
        for _, row in lex_df.iterrows():
            akkadian = str(row.get('form', '')).strip()
            english = str(row.get('norm', '')).strip()
            if akkadian and english and akkadian != 'nan' and english != 'nan':
                english = english.replace('_', ' ').lower()
                mappings[akkadian].add(english)
        
        logger.info(f'Loaded {len(mappings)} lexicon entries')
        return mappings
    
    def paraphrase_translation(self, translation: str) -> list:
        """Generate paraphrases via phrase substitution."""
        paraphrases = [translation]
        variations = {
            'of silver': ['of silver', 'in silver', 'from silver'],
            'minas of': ['minas of', 'mina(s) of', 'minas'],
            'seal of': ['seal of', 'the seal of', 'seal from'],
            'said:': ['stated:', 'said:', 'declared:', 'spoke:'],
            'he said': ['he said', 'saying', 'stating', 'he states'],
            'gave to': ['gave to', 'presented to', 'transferred to'],
            'the king': ['the king', 'king', 'his majesty'],
            'and': ['and', ',', 'plus'],
            'was': ['was', 'is', 'being'],
            'made': ['made', 'created', 'produced', 'crafted'],
        }
        current = translation
        for original, replacements in variations.items():
            if original in current:
                for replacement in replacements:
                    if replacement != original:
                        variant = current.replace(original, replacement, 1)
                        if variant != translation and len(variant) > 5:
                            paraphrases.append(variant)
        # Punctuation/formatting variations
        punct_variants = [
            translation.replace(':', ''),
            translation.replace(',', ';'),
        ]
        paraphrases.extend([p for p in punct_variants if p and p != translation])
        return list(set(paraphrases))[:8]
    
    def augment(self, train_df: pd.DataFrame, multiplier: float = 5.0) -> pd.DataFrame:
        """Augment training data by the given multiplier."""
        original_size = len(train_df)
        num_synthetic = int(original_size * multiplier) - original_size
        
        logger.info(f'Augmenting: {original_size} -> ~{original_size + num_synthetic} samples')
        synthetic_pairs = []
        
        # Strategy 1: Paraphrasing (50%)
        paraphrase_target = int(num_synthetic * 0.5)
        remaining = paraphrase_target
        for _, row in train_df.iterrows():
            paraphrases = self.paraphrase_translation(row['translation'])
            for para in paraphrases[1:]:
                synthetic_pairs.append({
                    'transliteration': row['transliteration'],
                    'translation': para
                })
                remaining -= 1
                if remaining <= 0:
                    break
            if remaining <= 0:
                break
        
        # Strategy 2: Translation variations (30%)
        variation_target = int(num_synthetic * 0.3)
        remaining = variation_target
        for _, row in train_df.sample(min(remaining, len(train_df)), random_state=42).iterrows():
            translation = row['translation']
            for var in [
                translation.replace('(...)', '[details omitted]'),
                translation.replace('...', '[continues]'),
                translation.replace('  ', ' '),
                translation.replace('[', '(').replace(']', ')'),
            ]:
                if var != translation and len(var) > 5:
                    synthetic_pairs.append({
                        'transliteration': row['transliteration'],
                        'translation': var
                    })
                    remaining -= 1
                    if remaining <= 0:
                        break
            if remaining <= 0:
                break
        
        # Strategy 3: Segment combination (20%)
        segment_target = int(num_synthetic * 0.2)
        for _ in range(min(segment_target, len(train_df))):
            row1 = train_df.sample(1).iloc[0]
            row2 = train_df.sample(1).iloc[0]
            combined_src = f"{row1['transliteration'][:40]} / {row2['transliteration'][:40]}".rstrip('/')
            combined_tgt = f"{row1['translation']}; {row2['translation']}"
            if len(combined_src) > 5 and len(combined_tgt) > 10:
                synthetic_pairs.append({
                    'transliteration': combined_src,
                    'translation': combined_tgt
                })
        
        synthetic_df = pd.DataFrame(synthetic_pairs)
        augmented = pd.concat([train_df[['transliteration', 'translation']], synthetic_df], ignore_index=True)
        augmented = augmented.sample(frac=1, random_state=42).reset_index(drop=True)
        
        logger.info(f'Augmentation complete: {len(augmented)} samples ({len(augmented)/original_size:.1f}x)')
        return augmented

In [ ]:
augmentor = DataAugmentor(INPUT_DIR)
augmented_df = augmentor.augment(train_df, multiplier=AUGMENT_MULTIPLIER)
print(f'Augmented dataset: {len(augmented_df)} samples')
augmented_df.head()

## 4. Tokenizer

In [ ]:
class SimpleTokenizer:
    """Word-level tokenizer with special tokens."""
    
    def __init__(self):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1, '<SOS>': 2, '<EOS>': 3}
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.next_idx = 4
    
    def build_vocab(self, texts, min_freq=1):
        word_freq = Counter()
        for text in texts:
            word_freq.update(str(text).split())
        for word, freq in word_freq.items():
            if freq >= min_freq and word not in self.word2idx:
                self.word2idx[word] = self.next_idx
                self.idx2word[self.next_idx] = word
                self.next_idx += 1
    
    def encode(self, text, max_len=180):
        words = str(text).split()
        indices = [self.word2idx.get(w, 1) for w in words[:max_len - 2]]
        indices = [2] + indices + [3]  # SOS + tokens + EOS
        while len(indices) < max_len:
            indices.append(0)
        return torch.tensor(indices[:max_len], dtype=torch.long)
    
    def decode(self, indices):
        words = []
        for idx in indices:
            if idx == 0 or idx == 3:  # PAD or EOS
                break
            if idx == 2:  # SOS
                continue
            words.append(self.idx2word.get(idx, '<UNK>'))
        return ' '.join(words)
    
    def __len__(self):
        return len(self.word2idx)


# Build tokenizers on augmented data
src_tokenizer = SimpleTokenizer()
tgt_tokenizer = SimpleTokenizer()
src_tokenizer.build_vocab(augmented_df['transliteration'].values)
tgt_tokenizer.build_vocab(augmented_df['translation'].values)

print(f'Source vocab: {len(src_tokenizer)} tokens')
print(f'Target vocab: {len(tgt_tokenizer)} tokens')

In [ ]:
# Encode all data
src_data = torch.stack([src_tokenizer.encode(t, MAX_LEN) for t in augmented_df['transliteration'].values])
tgt_data = torch.stack([tgt_tokenizer.encode(t, MAX_LEN) for t in augmented_df['translation'].values])

print(f'Source tensor: {src_data.shape}')
print(f'Target tensor: {tgt_data.shape}')

## 5. Model Architecture

In [ ]:
class AttentionLayer(nn.Module):
    """Bahdanau attention mechanism."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.query_proj = nn.Linear(hidden_dim, hidden_dim)
        self.key_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1)
    
    def forward(self, query, keys):
        if query.dim() == 1:
            query = query.unsqueeze(0)
        if keys.dim() == 2:
            keys = keys.unsqueeze(0)
        query_proj = self.query_proj(query).unsqueeze(1)
        key_proj = self.key_proj(keys)
        scores = torch.tanh(query_proj + key_proj)
        scores = self.v(scores).squeeze(-1)
        weights = torch.softmax(scores, dim=-1)
        context = (weights.unsqueeze(-1) * keys).sum(dim=1)
        return context, weights


class LabelSmoothingCrossEntropy(nn.Module):
    """Cross-entropy with label smoothing."""
    def __init__(self, num_classes, smoothing=0.1, ignore_index=0):
        super().__init__()
        self.smoothing = smoothing
        self.num_classes = num_classes
        self.ignore_index = ignore_index
    
    def forward(self, logits, targets):
        log_probs = torch.log_softmax(logits, dim=-1)
        mask = targets != self.ignore_index
        if not mask.any():
            return torch.tensor(0.0, device=logits.device)
        confidence = 1.0 - self.smoothing
        smooth_label = self.smoothing / self.num_classes
        with torch.no_grad():
            true_probs = torch.zeros_like(log_probs)
            true_probs.fill_(smooth_label)
            true_probs.scatter_(1, targets.unsqueeze(1), confidence)
        loss = torch.sum(-true_probs * log_probs, dim=-1)
        return loss[mask].mean()


def create_model(src_vocab_size, tgt_vocab_size, device):
    """Create all model components."""
    embedding = nn.Embedding(src_vocab_size, EMBEDDING_DIM).to(device)
    tgt_embedding = nn.Embedding(tgt_vocab_size, EMBEDDING_DIM).to(device)
    encoder_rnn = nn.LSTM(EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, batch_first=True,
                          dropout=LSTM_DROPOUT if NUM_LAYERS > 1 else 0).to(device)
    decoder_rnn = nn.LSTM(EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, batch_first=True,
                          dropout=LSTM_DROPOUT if NUM_LAYERS > 1 else 0).to(device)
    attention = AttentionLayer(HIDDEN_DIM).to(device)
    decoder_linear = nn.Linear(HIDDEN_DIM * 2, tgt_vocab_size).to(device)
    
    models = [embedding, tgt_embedding, encoder_rnn, decoder_rnn, attention, decoder_linear]
    total_params = sum(p.numel() for m in models for p in m.parameters())
    print(f'Model: {total_params:,} parameters ({total_params * 4 / 1e9:.2f} GB)')
    return models


# Create model
models = create_model(len(src_tokenizer), len(tgt_tokenizer), device)

## 6. Training Functions

In [ ]:
def train_epoch(models, optimizer, criterion, train_data, batch_size, device):
    """Train for one epoch."""
    embedding, tgt_embedding, encoder_rnn, decoder_rnn, attention, decoder = models
    for m in models:
        m.train()
    
    emb_dropout = nn.Dropout(EMBEDDING_DROPOUT).to(device)
    dec_dropout = nn.Dropout(DECODER_DROPOUT).to(device)
    
    src_data, tgt_data = train_data
    total_loss = 0
    batch_count = 0
    
    indices = torch.randperm(len(src_data))
    src_shuffled = src_data[indices]
    tgt_shuffled = tgt_data[indices]
    
    for batch_start in range(0, len(src_data), batch_size):
        batch_end = min(batch_start + batch_size, len(src_data))
        src_batch = src_shuffled[batch_start:batch_end].to(device)
        tgt_batch = tgt_shuffled[batch_start:batch_end].to(device)
        
        optimizer.zero_grad()
        
        # Encode
        embedded = emb_dropout(embedding(src_batch))
        encoder_out, (enc_hidden, enc_cell) = encoder_rnn(embedded)
        dec_hidden, dec_cell = enc_hidden, enc_cell
        
        # Decode step by step
        loss = 0
        num_content_steps = 0
        for step in range(1, tgt_batch.shape[1]):
            prev_token = tgt_batch[:, step - 1]
            target = tgt_batch[:, step]
            
            non_pad = (target != 0).sum().item()
            if non_pad == 0:
                break
            
            prev_embedded = emb_dropout(tgt_embedding(prev_token))
            _, (dec_hidden, dec_cell) = decoder_rnn(prev_embedded.unsqueeze(1), (dec_hidden, dec_cell))
            hidden_vec = dec_hidden[-1]
            
            context, _ = attention(hidden_vec, encoder_out)
            decoder_input = torch.cat([hidden_vec, context], dim=-1)
            logits = decoder(dec_dropout(decoder_input))
            
            loss += criterion(logits, target)
            num_content_steps += 1
        
        loss = loss / max(num_content_steps, 1)
        
        if torch.isnan(loss):
            optimizer.zero_grad()
            continue
        
        loss.backward()
        grad_params = [p for m in models for p in m.parameters()]
        torch.nn.utils.clip_grad_norm_(grad_params, max_norm=GRAD_CLIP)
        optimizer.step()
        
        total_loss += loss.item()
        batch_count += 1
    
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return total_loss / batch_count if batch_count > 0 else 0


def validate(models, criterion, val_data, batch_size, device):
    """Validate model."""
    embedding, tgt_embedding, encoder_rnn, decoder_rnn, attention, decoder = models
    for m in models:
        m.eval()
    
    src_data, tgt_data = val_data
    total_loss = 0
    batch_count = 0
    
    with torch.no_grad():
        for batch_start in range(0, len(src_data), batch_size):
            batch_end = min(batch_start + batch_size, len(src_data))
            src_batch = src_data[batch_start:batch_end].to(device)
            tgt_batch = tgt_data[batch_start:batch_end].to(device)
            
            embedded = embedding(src_batch)
            encoder_out, (enc_hidden, enc_cell) = encoder_rnn(embedded)
            dec_hidden, dec_cell = enc_hidden, enc_cell
            
            loss = 0
            num_content_steps = 0
            for step in range(1, tgt_batch.shape[1]):
                prev_token = tgt_batch[:, step - 1]
                target = tgt_batch[:, step]
                non_pad = (target != 0).sum().item()
                if non_pad == 0:
                    break
                prev_embedded = tgt_embedding(prev_token)
                _, (dec_hidden, dec_cell) = decoder_rnn(prev_embedded.unsqueeze(1), (dec_hidden, dec_cell))
                hidden_vec = dec_hidden[-1]
                context, _ = attention(hidden_vec, encoder_out)
                decoder_input = torch.cat([hidden_vec, context], dim=-1)
                logits = decoder(decoder_input)
                loss += criterion(logits, target)
                num_content_steps += 1
            
            loss = loss / max(num_content_steps, 1)
            if not torch.isnan(loss):
                total_loss += loss.item()
                batch_count += 1
    
    return total_loss / batch_count if batch_count > 0 else 0

## 7. K-Fold Training Loop

In [ ]:
def train_fold(fold_idx, train_indices, val_indices, src_data, tgt_data, device):
    """Train a single fold. Returns best val loss and saved checkpoint dict."""
    print(f'\n{"="*60}')
    print(f'FOLD {fold_idx + 1}/{NUM_FOLDS}')
    print(f'{"="*60}')
    
    train_src = src_data[train_indices]
    train_tgt = tgt_data[train_indices]
    val_src = src_data[val_indices]
    val_tgt = tgt_data[val_indices]
    
    print(f'Train: {len(train_src)}, Val: {len(val_src)}')
    
    # Fresh model for each fold
    fold_models = create_model(len(src_tokenizer), len(tgt_tokenizer), device)
    
    criterion = LabelSmoothingCrossEntropy(
        num_classes=len(tgt_tokenizer), smoothing=LABEL_SMOOTHING, ignore_index=0
    )
    
    optimizer = torch.optim.Adam(
        [p for m in fold_models for p in m.parameters()],
        lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    
    best_val_loss = float('inf')
    best_checkpoint = None
    patience_counter = 0
    annealing_mode = False
    epochs_in_annealing = 0
    
    for epoch in range(NUM_EPOCHS):
        train_loss = train_epoch(
            fold_models, optimizer, criterion,
            (train_src, train_tgt), BATCH_SIZE, device
        )
        val_loss = validate(
            fold_models, criterion,
            (val_src, val_tgt), BATCH_SIZE, device
        )
        
        ratio = val_loss / train_loss if train_loss > 0 else float('inf')
        
        ann_str = f' | Annealing: {epochs_in_annealing}/{EARLY_STOP_PATIENCE}' if annealing_mode else ''
        print(f'  Epoch {epoch+1:3d}/{NUM_EPOCHS} | '
              f'Train: {train_loss:.4f} | Val: {val_loss:.4f} | '
              f'Ratio: {ratio:.2f}x{ann_str}')
        
        # Save best
        if val_loss < best_val_loss and ratio <= OVERFITTING_THRESHOLD:
            best_val_loss = val_loss
            patience_counter = 0
            if annealing_mode:
                epochs_in_annealing = 0
            best_checkpoint = {
                'src_embedding': fold_models[0].state_dict(),
                'tgt_embedding': fold_models[1].state_dict(),
                'rnn': fold_models[2].state_dict(),
                'decoder_rnn': fold_models[3].state_dict(),
                'attention': fold_models[4].state_dict(),
                'decoder': fold_models[5].state_dict(),
            }
            # Save to disk immediately to free memory
            torch.save(best_checkpoint, OUTPUT_DIR / f'fold_{fold_idx}_best.pt')
            print(f'    -> Saved best (val_loss={val_loss:.4f})')
        else:
            patience_counter += 1
        
        # Overfitting detection
        if epoch >= 15 and ratio > OVERFITTING_THRESHOLD and not annealing_mode:
            annealing_mode = True
            epochs_in_annealing = 0
            current_lr = optimizer.param_groups[0]['lr']
            new_lr = current_lr * 0.5
            for pg in optimizer.param_groups:
                pg['lr'] = new_lr
            print(f'    OVERFITTING: LR {current_lr:.2e} -> {new_lr:.2e}')
        
        if annealing_mode:
            epochs_in_annealing += 1
            if epochs_in_annealing % 5 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                new_lr = current_lr * 0.7
                for pg in optimizer.param_groups:
                    pg['lr'] = new_lr
            if epochs_in_annealing >= EARLY_STOP_PATIENCE:
                print(f'    Early stopping after {EARLY_STOP_PATIENCE} annealing epochs')
                break
    
    # Clean up GPU
    del fold_models, optimizer
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    
    print(f'Fold {fold_idx + 1} best val loss: {best_val_loss:.4f}')
    return best_val_loss

In [ ]:
# Run K-Fold training (or single split if NUM_FOLDS=1)
if NUM_FOLDS == 1:
    print(f'Starting single train/test split (80/20)')
else:
    print(f'Starting {NUM_FOLDS}-fold cross-validation')
    
print(f'Dataset: {len(src_data)} samples, {NUM_EPOCHS} epochs max')
start_time = datetime.now()

fold_results = []
indices = np.arange(len(augmented_df))

if NUM_FOLDS == 1:
    # Single train/test split
    train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=SEED, shuffle=True)
    best_val = train_fold(0, train_idx, val_idx, src_data, tgt_data, device)
    fold_results.append({'fold': 1, 'val_loss': best_val})
else:
    # K-Fold cross-validation
    kfold = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(indices)):
        best_val = train_fold(fold_idx, train_idx, val_idx, src_data, tgt_data, device)
        fold_results.append({'fold': fold_idx + 1, 'val_loss': best_val})

# Summary
elapsed = datetime.now() - start_time
print(f'\n{"="*60}')
if NUM_FOLDS == 1:
    print(f'TRAINING COMPLETE ({elapsed})')
else:
    print(f'K-FOLD COMPLETE ({elapsed})')
print(f'{"="*60}')
for r in fold_results:
    print(f"  Fold {r['fold']}: Val Loss = {r['val_loss']:.4f}")
    
if NUM_FOLDS > 1:
    avg_loss = np.mean([r['val_loss'] for r in fold_results])
    std_loss = np.std([r['val_loss'] for r in fold_results])
    print(f'\nAverage: {avg_loss:.4f} +/- {std_loss:.4f}')

best_fold = min(fold_results, key=lambda x: x['val_loss'])
print(f'Best fold: {best_fold["fold"]} (loss={best_fold["val_loss"]:.4f})')

## 8. Inference — Beam Search

In [ ]:
def load_checkpoint(checkpoint_path, device):
    """Load model from checkpoint."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    
    src_vocab = checkpoint['src_embedding']['weight'].shape[0]
    tgt_vocab = checkpoint['decoder']['weight'].shape[0]
    embed_dim = checkpoint['src_embedding']['weight'].shape[1]
    
    # Detect hidden size and layers from encoder RNN
    rnn_state = checkpoint['rnn']
    num_layers = max(int(k.split('_l')[-1]) for k in rnn_state if '_l' in k) + 1
    hidden_size = rnn_state['weight_hh_l0'].shape[0] // 4
    
    embedding = nn.Embedding(src_vocab, embed_dim).to(device)
    tgt_embedding = nn.Embedding(tgt_vocab, embed_dim).to(device)
    encoder_rnn = nn.LSTM(embed_dim, hidden_size, num_layers, batch_first=True,
                          dropout=0.3 if num_layers > 1 else 0).to(device)
    decoder_rnn = nn.LSTM(embed_dim, hidden_size, num_layers, batch_first=True,
                          dropout=0.3 if num_layers > 1 else 0).to(device)
    attn = AttentionLayer(hidden_size).to(device)
    decoder_linear = nn.Linear(hidden_size * 2, tgt_vocab).to(device)
    
    embedding.load_state_dict(checkpoint['src_embedding'])
    tgt_embedding.load_state_dict(checkpoint['tgt_embedding'])
    encoder_rnn.load_state_dict(checkpoint['rnn'])
    if 'decoder_rnn' in checkpoint:
        decoder_rnn.load_state_dict(checkpoint['decoder_rnn'])
    else:
        decoder_rnn.load_state_dict(checkpoint['rnn'])  # Fallback: shared
    attn.load_state_dict(checkpoint['attention'])
    decoder_linear.load_state_dict(checkpoint['decoder'])
    
    for m in [embedding, tgt_embedding, encoder_rnn, decoder_rnn, attn, decoder_linear]:
        m.eval()
    
    return [embedding, tgt_embedding, encoder_rnn, decoder_rnn, attn, decoder_linear]


def beam_search_decode(models, src_tensor, tgt_tokenizer, device,
                       beam_width=5, max_len=180, temperature=1.0):
    """Beam search decoding."""
    embedding, tgt_embedding, encoder_rnn, decoder_rnn, attention, decoder = models
    tgt_vocab_size = len(tgt_tokenizer)
    
    with torch.no_grad():
        embedded = embedding(src_tensor.unsqueeze(0).to(device))
        encoder_outputs, (hidden, cell) = encoder_rnn(embedded)
        encoder_outputs = encoder_outputs.squeeze(0)  # (seq_len, hidden)
        
        # Beam: (log_prob, tokens, hidden, cell)
        beam = [(0.0, [2], hidden, cell)]  # Start with SOS
        completed = []
        
        for step in range(max_len - 1):
            candidates = []
            
            for log_prob, tokens, hid, cel in beam:
                if tokens[-1] == 3:  # EOS
                    completed.append((log_prob, tokens))
                    continue
                
                current_token = min(max(tokens[-1], 0), tgt_vocab_size - 1)
                current_emb = tgt_embedding(torch.tensor([current_token], device=device))
                _, (new_hid, new_cel) = decoder_rnn(current_emb.unsqueeze(1), (hid, cel))
                
                hidden_vec = new_hid[-1, 0] if new_hid.dim() == 3 else new_hid[-1]
                context, _ = attention(hidden_vec, encoder_outputs)
                
                # Ensure both tensors have same dimensions before concat
                if hidden_vec.dim() == 1 and context.dim() == 2:
                    hidden_vec = hidden_vec.unsqueeze(0)
                elif hidden_vec.dim() == 2 and context.dim() == 1:
                    context = context.unsqueeze(0)
                
                decoder_input = torch.cat([hidden_vec, context], dim=-1)
                logits = decoder(decoder_input) / temperature
                
                # If logits has batch dimension, squeeze it out
                if logits.dim() == 2 and logits.shape[0] == 1:
                    logits = logits.squeeze(0)
                
                log_probs = torch.log_softmax(logits, dim=-1)
                
                # Repetition penalty
                for prev_token in tokens[-5:]:
                    if prev_token not in [0, 1, 2, 3]:
                        log_probs[prev_token] -= 10.0
                
                k = min(beam_width, log_probs.shape[-1])
                top_lp, top_idx = torch.topk(log_probs, k)
                
                for lp, idx in zip(top_lp, top_idx):
                    candidates.append((
                        log_prob + lp.item(),
                        tokens + [idx.item()],
                        new_hid, new_cel
                    ))
            
            candidates.sort(reverse=True, key=lambda x: x[0])
            beam = candidates[:beam_width]
            
            if not beam or (completed and len(completed) >= beam_width):
                break
        
        all_seqs = completed + [(lp, tok) for lp, tok, _, _ in beam]
        all_seqs.sort(reverse=True, key=lambda x: x[0])
        
        if all_seqs:
            return all_seqs[0][1]
        return [2, 3]  # SOS + EOS fallback


print('Inference functions defined.')

## 9. Generate Predictions

In [ ]:
# Load the best fold checkpoint
best_fold_idx = best_fold['fold'] - 1
checkpoint_path = OUTPUT_DIR / f'fold_{best_fold_idx}_best.pt'
print(f'Loading best checkpoint: {checkpoint_path}')

inf_models = load_checkpoint(checkpoint_path, device)
print('Model loaded for inference.')

# Generate predictions for test set
predictions = []
print(f'\nGenerating predictions for {len(test_df)} test samples...')

for idx, row in test_df.iterrows():
    akkadian = str(row['transliteration'])
    src_tensor = src_tokenizer.encode(akkadian, MAX_LEN)
    
    tokens = beam_search_decode(
        inf_models, src_tensor, tgt_tokenizer, device,
        beam_width=BEAM_WIDTH, max_len=MAX_LEN, temperature=BEAM_TEMPERATURE
    )
    
    translation = tgt_tokenizer.decode(tokens)
    predictions.append({'id': row['id'], 'translation': translation})
    print(f'  [{row["id"]}] {akkadian[:60]}...')
    print(f'       -> {translation[:80]}...')

print(f'\nGenerated {len(predictions)} predictions.')

In [ ]:
# Save submission
submission_df = pd.DataFrame(predictions)
submission_path = OUTPUT_DIR / 'submission.csv'
submission_df.to_csv(submission_path, index=False)

print(f'Submission saved to: {submission_path}')
print(f'Shape: {submission_df.shape}')
print(f'Columns: {list(submission_df.columns)}')
print(f'\n--- submission.csv ---')
print(submission_df.to_string())

In [ ]:
# Verify submission format
sample_sub = pd.read_csv(INPUT_DIR / 'sample_submission.csv')
assert list(submission_df.columns) == list(sample_sub.columns), \
    f'Column mismatch: {list(submission_df.columns)} vs {list(sample_sub.columns)}'
assert len(submission_df) == len(sample_sub), \
    f'Row count mismatch: {len(submission_df)} vs {len(sample_sub)}'
assert submission_df['id'].dtype == sample_sub['id'].dtype, \
    f'ID type mismatch: {submission_df["id"].dtype} vs {sample_sub["id"].dtype}'

print('Submission format verified!')
print(f'\nTotal runtime: {datetime.now() - start_time}')

# Cleanup fold checkpoints
for f in OUTPUT_DIR.glob('fold_*_best.pt'):
    f.unlink()
    print(f'Cleaned up: {f}')

# Record end time
notebook_end_time = datetime.now()
with open(str(OUTPUT_DIR / 'time.txt'), 'a') as f:
with open('time.txt', 'a') as f:
    f.write(f'End Time: {notebook_end_time.strftime("%Y-%m-%d %H:%M:%S")}\n')
    f.write(f'Total Duration: {total_duration}\n')

print('\n' + '='*60)
print(f'NOTEBOOK END TIME: {notebook_end_time.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'TOTAL DURATION: {total_duration}')
print('='*60)
print('\nDone!')
